<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EC%96%91%EA%B7%BC%EC%98%81%ED%95%99%EC%83%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# data.yaml 예시
path: ./datasets
train: images/train
val: images/val

nc: 2
names: ['lane', 'traffic_sign']

datasets/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/

model.train(
    data="/content/dataset/dataset.yaml",  # yaml 경로
    epochs=120,        # 학습 에폭 수
    imgsz=640,         # 입력 이미지 크기
    batch=16,          # 배치 사이즈
    name="lane_model",  # 저장 폴더 이름
)

from ultralytics import YOLO

# 모델 로드
model = YOLO('runs/detect/train/weights/best.pt')

# 이미지에 대한 예측
results = model.predict(source='sample.jpg')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 마운트 확인
import os
print("드라이브 내용:")
os.listdir('/content/drive/MyDrive/')

In [ ]:
# ZIP 파일을 코랩으로 복사
!cp "/content/drive/MyDrive/6_23_Lesson/dataset.zip" "/content/"

# 압축 해제
!unzip -o /content/dataset.zip -d /content/

# 압축 해제 확인
!ls -la /content/

In [ ]:
# 1. ultralytics 설치
!pip install ultralytics

# 2. 설치 확인 후 다시 실행
from ultralytics import YOLO
import glob
import os

print("✅ ultralytics 설치 완료!")

# 이미 학습된 모델 사용
model = YOLO('/content/dataset/best.pt')

# YouTube 영상 다운로드
!pip install yt-dlp
!yt-dlp -f 'best[height<=720]' -o '/content/test_video.%(ext)s' 'https://www.youtube.com/watch?v=AxLmroTo3rQ'

# 다운로드된 파일 찾기 (확장자가 다를 수 있음)
video_files = glob.glob('/content/test_video.*')
if video_files:
    video_path = video_files[0]
    print(f"📹 다운로드된 영상: {video_path}")

    # 추론 실행
    results = model(video_path)

    # 결과 표시 (영상의 경우 첫 번째 프레임만)
    if results:
        results[0].show()
else:
    print("❌ 영상 다운로드 실패")

# 기존 검증 데이터로 성능 측정
print("\n📊 모델 성능 평가:")
metrics = model.val(data='/content/dataset/dataset.yaml')
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

print("\n✅ 모든 작업 완료!")

vs code에서 data.yaml파일 확인하니 이름이 다르고 경로가 달라서 수정을 했다.

In [ ]:
# 1. dataset.yaml 파일 내용 확인
print("📋 dataset.yaml 파일 내용:")
with open('/content/dataset/dataset.yaml', 'r') as f:
    yaml_content = f.read()
    print(yaml_content)

In [ ]:
!pip install ultralytics yt-dlp

from ultralytics import YOLO
import glob

# yaml 수정 (핵심 문제 해결)
yaml_fix = '''path: /content/dataset
train: train/images
val: valid/images
names:
  0: lane
  1: traffic_sign
nc: 2'''

with open('/content/dataset/best.pt', 'w') as f:
    f.write(yaml_fix)

# 모델 로드 & 영상 다운로드 & 추론
model = YOLO('/content/dataset/best.pt')
!yt-dlp -f 'best[height<=720]' -o '/content/test_video.%(ext)s' 'https://www.youtube.com/watch?v=AxLmroTo3rQ'

video_path = glob.glob('/content/test_video.*')[0]
results = model(video_path)
results[0].show()

# 성능 평가
metrics = model.val(data='/content/dataset/dataset.yaml')
print(f"mAP50: {metrics.box.map50:.4f}")

In [ ]:
# 모델 검증 실행
metrics = model.val(data='/content/dataset/dataset.yaml')
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

기본 YOLO + 커스텀 YOLO 동시 사용
즉 전이학습하면서 80개의 클래스가 2개로 줄었으므로 확대해서 추가
🎯 Ultralytics의 실체:

기반: PyTorch 위에 구축된 래퍼(Wrapper)

내부: PyTorch 모델을 사용하지만 사용자에게는 숨김

장점: 복잡한 PyTorch 코드를 간단한 API로 제공

사용자 코드: model = YOLO('yolo11n.pt')
     ↓
Ultralytics: 내부적으로 PyTorch 모델 로드
     ↓  
실제 추론: PyTorch 텐서 연산
     ↓
결과 반환: 사용자 친화적 형태로 변환

In [ ]:
!pip install ultralytics yt-dlp

from ultralytics import YOLO
import glob
import cv2
import numpy as np
from IPython.display import Video
import shutil

# yaml 수정 (핵심 문제 해결)
yaml_fix = '''path: /content/dataset
train: train/images
val: valid/images
names:
  0: lane
  1: traffic_sign
nc: 2'''

with open('/content/dataset/dataset.yaml', 'w') as f:
    f.write(yaml_fix)

# 모델들 로드
print("🤖 모델 로드 중...")
base_model = YOLO('yolo11n.pt')  # 기본 YOLO (80개 클래스)
custom_model = YOLO('/content/dataset/best.pt')  # 커스텀 YOLO (2개 클래스)

print(f"기본 모델 클래스 수: {len(base_model.names)}")
print(f"커스텀 모델 클래스 수: {len(custom_model.names)}")

# 영상 다운로드
print("📥 YouTube 영상 다운로드 중...")
!yt-dlp -f 'best[height<=720]' -o '/content/test_video.%(ext)s' 'https://www.youtube.com/watch?v=AxLmroTo3rQ'

video_path = glob.glob('/content/test_video.*')[0]
print(f"✅ 다운로드 완료: {video_path}")

# 결합된 추론 함수
def combined_inference(video_path, output_path='/content/combined_result.mp4'):
    """기본 YOLO + 커스텀 YOLO 결과를 결합하여 영상 생성"""

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 출력 영상 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"🎬 영상 처리 중... (총 {total_frames} 프레임)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 기본 YOLO 추론 (차량, 사람, 신호등 등)
        base_results = base_model(frame, verbose=False)

        # 커스텀 YOLO 추론 (차선, 교통표지판)
        custom_results = custom_model(frame, verbose=False)

        # 결과 시각화
        annotated_frame = frame.copy()

        # 기본 YOLO 결과 그리기 (파란색)
        if base_results[0].boxes is not None:
            for box in base_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:  # 신뢰도 30% 이상만
                    label = f"{base_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)  # 파란색
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # 커스텀 YOLO 결과 그리기 (빨간색)
        if custom_results[0].boxes is not None:
            for box in custom_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:  # 신뢰도 30% 이상만
                    label = f"{custom_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # 빨간색
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 30 == 0:  # 30프레임마다 진행상황 출력
            print(f"   처리 중... {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")

    cap.release()
    out.release()
    print(f"✅ 결합 결과 영상 저장: {output_path}")

# 개별 추론 결과
print("\n🔍 개별 모델 추론 결과:")

# 커스텀 모델만으로 추론
print("1️⃣ 커스텀 모델 (차선, 교통표지판만):")
custom_results = custom_model(video_path, save=True, project='/content', name='custom_only')

# 기본 모델만으로 추론
print("2️⃣ 기본 모델 (일반 객체들):")
base_results = base_model(video_path, save=True, project='/content', name='base_only')

# 결합된 추론
print("3️⃣ 결합 모델 (모든 객체):")
combined_inference(video_path, '/content/combined_result.mp4')

# 결과 영상들을 표준 이름으로 복사
result_files = {
    'custom_result.mp4': glob.glob('/content/custom_only/*.avi') + glob.glob('/content/custom_only/*.mp4'),
    'base_result.mp4': glob.glob('/content/base_only/*.avi') + glob.glob('/content/base_only/*.mp4'),
    'final_combined_result.mp4': ['/content/combined_result.mp4']
}

print("\n📁 결과 파일들:")
for name, files in result_files.items():
    if files and files[0]:
        shutil.copy(files[0], f'/content/{name}')
        print(f"✅ {name} 생성 완료")

# 성능 평가 (커스텀 모델)
print("\n📊 커스텀 모델 성능 평가:")
metrics = custom_model.val(data='/content/dataset/dataset.yaml')
print(f"mAP50: {metrics.box.map50:.4f}")

# 최종 결과 영상 재생
print("\n🎬 최종 결합 결과 영상:")
Video('/content/final_combined_result.mp4', width=800)

print("\n🎯 결과 요약:")
print("🔵 파란색 박스: 기본 YOLO (차량, 사람, 신호등, 표지판 등)")
print("🔴 빨간색 박스: 커스텀 YOLO (차선, 교통표지판)")
print("\n💾 다운로드 가능한 파일들:")
print("- custom_result.mp4: 커스텀 모델만")
print("- base_result.mp4: 기본 모델만")
print("- final_combined_result.mp4: 모든 객체 탐지")

영상에 fps결과 추가

In [ ]:
!pip install ultralytics yt-dlp

from ultralytics import YOLO
import glob
import cv2
import numpy as np
from IPython.display import Video
import shutil
import time

# yaml 수정 (핵심 문제 해결)
yaml_fix = '''path: /content/dataset
train: train/images
val: valid/images
names:
  0: lane
  1: traffic_sign
nc: 2'''

with open('/content/dataset/dataset.yaml', 'w') as f:
    f.write(yaml_fix)

# 모델들 로드 (안전한 로드)
print("🤖 모델 로드 중...")
base_model = YOLO('yolo11n.pt')  # 기본 YOLO (80개 클래스)

# 커스텀 모델 안전 로드
try:
    custom_model = YOLO('/content/dataset/best.pt')  # 커스텀 YOLO (2개 클래스)
    print(f"✅ 커스텀 모델 로드 성공")
except:
    print("⚠️ 커스텀 모델 로드 실패. 기본 모델을 사용합니다.")
    custom_model = base_model

print(f"기본 모델 클래스 수: {len(base_model.names)}")
print(f"커스텀 모델 클래스 수: {len(custom_model.names)}")

# 영상 다운로드
print("📥 YouTube 영상 다운로드 중...")
!yt-dlp -f 'best[height<=720]' -o '/content/test_video.%(ext)s' 'https://www.youtube.com/watch?v=AxLmroTo3rQ'

video_path = glob.glob('/content/test_video.*')[0]
print(f"✅ 다운로드 완료: {video_path}")

# FPS 표시가 포함된 결합 추론 함수
def combined_inference(video_path, output_path='/content/combined_result.mp4'):
    """기본 YOLO + 커스텀 YOLO 결과를 결합하여 영상 생성 (FPS 표시 포함)"""

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 출력 영상 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # FPS 계산을 위한 변수들
    prev_time = time.time()
    fps_history = []
    fps_display = 0.0
    start_time = time.time()

    print(f"🎬 영상 처리 중... (총 {total_frames} 프레임, 원본 FPS: {fps})")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 추론 시간 측정 시작
        inference_start = time.time()

        # 기본 YOLO 추론 (차량, 사람, 신호등 등)
        base_results = base_model(frame, verbose=False)

        # 커스텀 YOLO 추론 (차선, 교통표지판)
        custom_results = custom_model(frame, verbose=False)

        # 추론 시간 계산
        inference_time = time.time() - inference_start

        # 결과 시각화
        annotated_frame = frame.copy()

        # 기본 YOLO 결과 그리기 (파란색)
        if base_results[0].boxes is not None:
            for box in base_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:  # 신뢰도 30% 이상만
                    label = f"{base_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)  # 파란색
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # 커스텀 YOLO 결과 그리기 (빨간색)
        if custom_results[0].boxes is not None:
            for box in custom_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:  # 신뢰도 30% 이상만
                    label = f"{custom_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # 빨간색
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # === FPS 계산 및 표시 (새로 추가된 부분) ===
        current_time = time.time()

        # 전체 평균 FPS 계산
        overall_fps = frame_count / (current_time - start_time) if frame_count > 0 else 0

        # 실시간 FPS 계산 (더 부드러운 표시)
        if frame_count > 0:
            frame_fps = 1.0 / (current_time - prev_time)
            fps_history.append(frame_fps)

            # 최근 10프레임의 평균 (너무 많으면 반응이 느려짐)
            if len(fps_history) > 10:
                fps_history.pop(0)
            fps_display = sum(fps_history) / len(fps_history)

        prev_time = current_time

        # 정보 텍스트 준비
        fps_text = f"FPS: {fps_display:.1f}"
        avg_fps_text = f"Avg: {overall_fps:.1f}"
        inference_text = f"Inference: {inference_time*1000:.0f}ms"
        progress_text = f"Frame: {frame_count}/{total_frames}"

        # 반투명 배경 생성 (정보 표시용)
        overlay = annotated_frame.copy()
        cv2.rectangle(overlay, (10, 10), (320, 120), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, annotated_frame, 0.3, 0, annotated_frame)

        # 정보 텍스트 표시
        cv2.putText(annotated_frame, fps_text, (20, 35),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)  # 초록색
        cv2.putText(annotated_frame, avg_fps_text, (20, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)  # 청록색
        cv2.putText(annotated_frame, inference_text, (20, 85),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)  # 노란색
        cv2.putText(annotated_frame, progress_text, (20, 105),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)  # 흰색

        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 30 == 0:  # 30프레임마다 진행상황 출력
            print(f"   처리 중... {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%) - 현재 FPS: {fps_display:.1f}")

    cap.release()
    out.release()

    # 최종 통계 출력
    total_time = time.time() - start_time
    final_avg_fps = frame_count / total_time
    print(f"✅ 결합 결과 영상 저장: {output_path}")
    print(f"📊 처리 완료 - 총 시간: {total_time:.1f}초, 평균 FPS: {final_avg_fps:.2f}")

# 개별 모델 추론 (FPS 표시 포함)
def single_model_inference(model, video_path, output_path, model_name, color):
    """단일 모델 추론 with FPS 표시"""

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_time = time.time()
    prev_time = time.time()
    fps_history = []

    print(f"🎬 {model_name} 모델 처리 중...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 추론 시간 측정
        inference_start = time.time()
        results = model(frame, verbose=False)
        inference_time = time.time() - inference_start

        annotated_frame = frame.copy()

        # 탐지 결과 그리기
        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:
                    label = f"{model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # FPS 계산
        current_time = time.time()
        overall_fps = frame_count / (current_time - start_time) if frame_count > 0 else 0

        if frame_count > 0:
            frame_fps = 1.0 / (current_time - prev_time)
            fps_history.append(frame_fps)
            if len(fps_history) > 10:
                fps_history.pop(0)

        current_fps = sum(fps_history) / len(fps_history) if fps_history else 0
        prev_time = current_time

        # 정보 표시
        fps_text = f"{model_name} FPS: {current_fps:.1f}"
        inference_text = f"Inference: {inference_time*1000:.0f}ms"

        # 배경
        overlay = annotated_frame.copy()
        cv2.rectangle(overlay, (10, 10), (300, 80), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, annotated_frame, 0.3, 0, annotated_frame)

        # 텍스트
        cv2.putText(annotated_frame, fps_text, (20, 35),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(annotated_frame, inference_text, (20, 60),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 30 == 0:
            print(f"   {model_name}: {frame_count}/{total_frames} - FPS: {current_fps:.1f}")

    cap.release()
    out.release()

    total_time = time.time() - start_time
    final_fps = frame_count / total_time
    print(f"✅ {model_name} 완료: {output_path} (평균 FPS: {final_fps:.2f})")

    return output_path

# 개별 추론 결과
print("\n🔍 개별 모델 추론 결과:")

# 커스텀 모델만으로 추론
print("1️⃣ 커스텀 모델 (차선, 교통표지판만):")
custom_result_path = single_model_inference(
    custom_model, video_path, '/content/custom_with_fps.mp4',
    'Custom', (0, 0, 255)  # 빨간색
)

# 기본 모델만으로 추론
print("2️⃣ 기본 모델 (일반 객체들):")
base_result_path = single_model_inference(
    base_model, video_path, '/content/base_with_fps.mp4',
    'Base', (255, 0, 0)  # 파란색
)

# 결합된 추론
print("3️⃣ 결합 모델 (모든 객체):")
combined_inference(video_path, '/content/combined_with_fps.mp4')

# 기존 방식도 유지 (호환성)
print("\n📁 기존 방식 결과도 생성...")
custom_results = custom_model(video_path, save=True, project='/content', name='custom_only')
base_results = base_model(video_path, save=True, project='/content', name='base_only')

# 결과 파일들 정리
result_files = {
    'custom_result.mp4': glob.glob('/content/custom_only/*.avi') + glob.glob('/content/custom_only/*.mp4'),
    'base_result.mp4': glob.glob('/content/base_only/*.avi') + glob.glob('/content/base_only/*.mp4'),
    'final_combined_result.mp4': ['/content/combined_with_fps.mp4']
}

print("\n📁 결과 파일들:")
for name, files in result_files.items():
    if files and files[0]:
        if os.path.exists(files[0]):
            shutil.copy(files[0], f'/content/{name}')
            print(f"✅ {name} 생성 완료")

# 성능 평가 (커스텀 모델)
if custom_model != base_model:
    try:
        print("\n📊 커스텀 모델 성능 평가:")
        metrics = custom_model.val(data='/content/dataset/dataset.yaml')
        print(f"mAP50: {metrics.box.map50:.4f}")
    except Exception as e:
        print(f"⚠️ 성능 평가 실패: {e}")

# 최종 결과 영상 재생
print("\n🎬 최종 결합 결과 영상 (FPS 표시 포함):")
Video('/content/final_combined_result.mp4', width=800)

print("\n🎯 결과 요약:")
print("🔵 파란색 박스: 기본 YOLO (차량, 사람, 신호등, 표지판 등)")
print("🔴 빨간색 박스: 커스텀 YOLO (차선, 교통표지판)")
print("🟢 초록색 숫자: 실시간 FPS")
print("🟡 노란색 숫자: 추론 시간")
print("\n💾 다운로드 가능한 파일들:")
print("- custom_with_fps.mp4: 커스텀 모델 (FPS 표시)")
print("- base_with_fps.mp4: 기본 모델 (FPS 표시)")
print("- combined_with_fps.mp4: 결합 모델 (FPS 표시)")
print("- custom_result.mp4: 커스텀 모델 (기존 방식)")
print("- base_result.mp4: 기본 모델 (기존 방식)")
print("- final_combined_result.mp4: 최종 결합 결과")

⚡ TensorRT 변환 과정:

PyTorch → TensorRT: .export(format='engine') 자동 변환
엔진 파일 생성: .engine 파일로 저장
최적화된 추론: 기존 API 그대로 사용

📊 성능 비교 기능:

실시간 속도 측정: 프레임별 처리 시간
PyTorch vs TensorRT: 직접 비교
속도 향상 비율: 몇 배 빨라졌는지 측정

🎯 예상 성능 개선:

추론 속도: 2-5배 향상
실시간 처리: 30 FPS → 60+ FPS
GPU 메모리: 30-50% 절약

🎬 최종 결과:

tensorrt_final_result.mp4: TensorRT로 최적화된 완전한 추론 결과
영상에 실시간 FPS 표시
모든 객체 탐지 (기본 80개 + 커스텀 2개)

이제 같은 결과를 훨씬 빠르게 얻을 수 있습니다! 🔥재시도!pip install ultralytics yt-dlp

from ultralytics import YOLO
import glob
import cv2
import numpy as np
from IPython.display import Video
import shutil

# yaml 수정 (핵심 문제 해결)
yaml_fix = '''path: /content/dataset
train: train/images
val: valid/images
names:
  0: lane
  1: traffic_sign
pastedSS이  코드는 파이토치도 아니지?편집맞습니다! 이 코드는 PyTorch가 아닙니다! 🎯
🔍 현재 코드의 정체:
이것은 Ultralytics YOLO입니다:
pythonfrom ultralytics import YOLO  # ← Ultralytics 라이브러리
model = YOLO('yolo11n.pt')    # ← YOLOv11 모델
📊 각 프레임워크 비교:
🐍 순수 PyTorch:
pythonimport torch
import torch.nn as nn
model = torch.load('model.pth')
output = model(input_tensor)
🔥 Ultralytics YOLO:
pythonfrom ultralytics import YOLO  # ← 현재 사용 중
model = YOLO('yolo11n.pt')
results = model(image)
⚡ TensorRT:
pythonimport tensorrt as trt
# C++/CUDA 기반 추론 엔진
🎯 Ultralytics의 실체:

기반: PyTorch 위에 구축된 래퍼(Wrapper)
내부: PyTorch 모델을 사용하지만 사용자에게는 숨김
장점: 복잡한 PyTorch 코드를 간단한 API로 제공

🔄 실제 처리 과정:
사용자 코드: model = YOLO('yolo11n.pt')
     ↓
Ultralytics: 내부적으로 PyTorch 모델 로드
     ↓  
실제 추론: PyTorch 텐서 연산
     ↓
결과 반환: 사용자 친화적 형태로 변환
결론: 겉보기에는 간단해 보이지만, 실제로는 PyTorch 기반입니다!
Ultralytics가 복잡한 PyTorch 코드를 사용하기 쉽게 포장해서 제공하는 것이죠! 🎁재시도Claude는 실수를 할 수 있습니다. 응답을 반드시 다시 확인해 주세요. Sonnet 4

-------------------------------------------------
Loading yolo11n.engine for TensorRT inference...

Loading /content/dataset/best.engine for TensorRT inference...

🐍 PyTorch 평균: 62.98ms/frame (15.9 FPS)
⚡ TensorRT 평균: 11.83ms/frame (84.5 FPS)
🚀 속도 향상: 5.32x

🔥 TensorRT 최적화된 결합 추론 실행...

🎬 TensorRT 최적화 영상 처리 중... (총 3990 프레임)

   처리 중... 50/3990 (1.3%) - 평균 94.4 FPS

   처리 중... 100/3990 (2.5%) - 평균 93.9 FPS

   처리 중... 150/3990 (3.8%) - 평균 91.5 FPS

   처리 중... 200/3990 (5.0%) - 평균 89.5 FPS

   처리 중... 250/3990 (6.3%) - 평균 88.5 FPS

   처리 중... 300/3990 (7.5%) - 평균 85.7 FPS

   처리 중... 350/3990 (8.8%) - 평균 83.0 FPS

   처리 중... 400/3990 (10.0%) - 평균 80.3 FPS

   처리 중... 450/3990 (11.3%) - 평균 78.7 FPS

   처리 중... 500/3990 (12.5%) - 평균 78.3 FPS
   
   처리 중... 550/3990 (13.8%) - 평균 78.0 FPS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ZIP 파일을 코랩으로 복사
!cp "/content/drive/MyDrive/6_23_Lesson/dataset.zip" "/content/"

# 압축 해제
!unzip -o /content/dataset.zip -d /content/

# 압축 해제 확인
!ls -la /content/

In [ ]:
!pip install ultralytics yt-dlp

from ultralytics import YOLO
import glob
import cv2
import numpy as np
from IPython.display import Video
import shutil
import time

# yaml 수정 (핵심 문제 해결)
yaml_fix = '''path: /content/dataset
train: train/images
val: valid/images
names:
  0: lane
  1: traffic_sign
nc: 2'''

with open('/content/dataset/dataset_fixed.yaml', 'w') as f:
    f.write(yaml_fix)

print("🚀 TensorRT 최적화 YOLO 추론 시작!")
print("="*60)

# 1️⃣ 기본 모델들 로드
print("🤖 기본 모델 로드 중...")
base_model = YOLO('yolo11n.pt')
custom_model = YOLO('/content/dataset/best.pt')

print(f"기본 모델 클래스 수: {len(base_model.names)}")
print(f"커스텀 모델 클래스 수: {len(custom_model.names)}")

# 2️⃣ TensorRT로 변환
print("\n⚡ TensorRT 변환 중...")
print("기본 모델 → TensorRT 변환...")
base_model.export(format='engine', half=True, device=0)  # FP16 최적화
base_trt_path = 'yolo11n.engine'

print("커스텀 모델 → TensorRT 변환...")
custom_model.export(format='engine', half=True, device=0)
custom_trt_path = '/content/dataset/best.engine'

# 3️⃣ TensorRT 모델 로드
print("\n🔥 TensorRT 모델 로드 중...")
base_trt_model = YOLO(base_trt_path)
custom_trt_model = YOLO(custom_trt_path)

print("✅ TensorRT 모델 로드 완료!")

# 4️⃣ 영상 다운로드
print("\n📥 YouTube 영상 다운로드 중...")
!yt-dlp -f 'best[height<=720]' -o '/content/test_video.%(ext)s' 'https://www.youtube.com/watch?v=AxLmroTo3rQ'

video_path = glob.glob('/content/test_video.*')[0]
print(f"✅ 다운로드 완료: {video_path}")

# 5️⃣ 성능 비교 함수
def performance_comparison(video_path, frames_to_test=100):
    """PyTorch vs TensorRT 성능 비교"""

    print(f"\n⏱️ 성능 비교 (첫 {frames_to_test}프레임)")
    print("-" * 50)

    cap = cv2.VideoCapture(video_path)

    # PyTorch 모델 성능 테스트
    pytorch_times = []
    for i in range(frames_to_test):
        ret, frame = cap.read()
        if not ret:
            break

        start_time = time.time()
        _ = base_model(frame, verbose=False)
        _ = custom_model(frame, verbose=False)
        end_time = time.time()

        pytorch_times.append(end_time - start_time)

    # TensorRT 모델 성능 테스트
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # 처음으로 되돌리기
    tensorrt_times = []
    for i in range(frames_to_test):
        ret, frame = cap.read()
        if not ret:
            break

        start_time = time.time()
        _ = base_trt_model(frame, verbose=False)
        _ = custom_trt_model(frame, verbose=False)
        end_time = time.time()

        tensorrt_times.append(end_time - start_time)

    cap.release()

    # 결과 출력
    pytorch_avg = np.mean(pytorch_times) * 1000  # ms로 변환
    tensorrt_avg = np.mean(tensorrt_times) * 1000
    speedup = pytorch_avg / tensorrt_avg

    print(f"🐍 PyTorch 평균: {pytorch_avg:.2f}ms/frame ({1000/pytorch_avg:.1f} FPS)")
    print(f"⚡ TensorRT 평균: {tensorrt_avg:.2f}ms/frame ({1000/tensorrt_avg:.1f} FPS)")
    print(f"🚀 속도 향상: {speedup:.2f}x")

    return speedup

# 성능 비교 실행
speedup_ratio = performance_comparison(video_path)

# 6️⃣ TensorRT 최적화된 결합 추론
def tensorrt_combined_inference(video_path, output_path='/content/tensorrt_result.mp4'):
    """TensorRT 최적화된 결합 추론"""

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 출력 영상 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"\n🎬 TensorRT 최적화 영상 처리 중... (총 {total_frames} 프레임)")

    frame_count = 0
    total_inference_time = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # TensorRT 추론 (시간 측정)
        start_time = time.time()

        # 기본 TensorRT 모델 추론
        base_results = base_trt_model(frame, verbose=False)

        # 커스텀 TensorRT 모델 추론
        custom_results = custom_trt_model(frame, verbose=False)

        inference_time = time.time() - start_time
        total_inference_time += inference_time

        # 결과 시각화
        annotated_frame = frame.copy()

        # 기본 YOLO 결과 그리기 (파란색)
        if base_results[0].boxes is not None:
            for box in base_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:
                    label = f"{base_trt_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

        # 커스텀 YOLO 결과 그리기 (빨간색)
        if custom_results[0].boxes is not None:
            for box in custom_results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls = int(box.cls[0])

                if conf > 0.3:
                    label = f"{custom_trt_model.names[cls]} {conf:.2f}"
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(annotated_frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # TensorRT 정보 표시
        fps_text = f"TensorRT: {1/inference_time:.1f} FPS"
        cv2.putText(annotated_frame, fps_text, (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 50 == 0:
            avg_fps = frame_count / total_inference_time
            print(f"   처리 중... {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%) - 평균 {avg_fps:.1f} FPS")

    cap.release()
    out.release()

    avg_fps = frame_count / total_inference_time
    print(f"✅ TensorRT 결과 영상 저장: {output_path}")
    print(f"📊 평균 처리 속도: {avg_fps:.1f} FPS")

    return avg_fps

# 7️⃣ TensorRT 최적화된 추론 실행
print("\n🔥 TensorRT 최적화된 결합 추론 실행...")
tensorrt_fps = tensorrt_combined_inference(video_path, '/content/tensorrt_final_result.mp4')

# 8️⃣ 기존 PyTorch 추론도 실행 (비교용)
print("\n🐍 PyTorch 기존 추론 (비교용)...")
def pytorch_combined_inference(video_path, output_path='/content/pytorch_result.mp4'):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    start_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        base_results = base_model(frame, verbose=False)
        custom_results = custom_model(frame, verbose=False)

        # 간단한 시각화 (속도 비교용)
        annotated_frame = frame.copy()
        cv2.putText(annotated_frame, "PyTorch", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

        out.write(annotated_frame)
        frame_count += 1

        if frame_count >= 100:  # 100프레임만 처리 (비교용)
            break

    cap.release()
    out.release()

    total_time = time.time() - start_time
    pytorch_fps = frame_count / total_time
    return pytorch_fps

pytorch_fps = pytorch_combined_inference(video_path)

# 9️⃣ 성능 평가 (커스텀 모델)
print("\n📊 커스텀 모델 성능 평가:")
metrics = custom_model.val(data='/content/dataset/dataset_fixed.yaml')  # custom_trt_model 대신 custom_model
print(f"mAP50: {metrics.box.map50:.4f}")

# 🔟 최종 결과 및 비교
print("\n" + "="*60)
print("🎯 최종 성능 비교 결과:")
print(f"🐍 PyTorch: {pytorch_fps:.1f} FPS")
print(f"⚡ TensorRT: {tensorrt_fps:.1f} FPS")
print(f"🚀 전체 속도 향상: {tensorrt_fps/pytorch_fps:.2f}x")

print(f"\n📊 모델 정확도 (mAP50): {metrics.box.map50:.4f}")

print("\n🎬 최종 TensorRT 결과 영상:")
Video('/content/tensorrt_final_result.mp4', width=800)

print("\n🎉 TensorRT 최적화 완료!")
print("🔵 파란색 박스: 기본 YOLO 객체들 (TensorRT 최적화)")
print("🔴 빨간색 박스: 커스텀 객체들 (TensorRT 최적화)")
print("💚 초록색 텍스트: 실시간 FPS 표시")

print("\n💾 생성된 파일들:")
print("- tensorrt_final_result.mp4: TensorRT 최적화된 최종 결과")
print("- pytorch_result.mp4: PyTorch 비교용 결과")
print("- yolo11n.engine: 기본 모델 TensorRT 엔진")
print("- best.engine: 커스텀 모델 TensorRT 엔진")